# <center>Блок 6. Математика в ML. Часть II<center>
## <center>MATH&ML-9. Математика ансамблевых методов<center>
### <center>1.Введение<center>
### <center>2.Ансамбли моделей. Бустреппинг. Бэггинг<center>
#### <center>Bias и Variance<center>
#### <center>Бэггинг<center>

In [132]:
import pandas as pd
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn import tree, metrics
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import BaggingClassifier, RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error
import numpy as np
from sklearn.model_selection import GridSearchCV

In [17]:
data = pd.read_csv('C:\\IDE\\data\\Block_6\\wineQualityReds.csv')
data.head()

,Unnamed: 0,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality
0,1,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,2,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,3,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,4,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,5,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [18]:
data['quality'] = data['quality'].apply(lambda x: 1 if x >= 6 else 0)

In [19]:
X = data.drop('quality', axis=1)
y = data['quality']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

In [20]:
# Задание 2.7

lg_model = LogisticRegression(random_state=42)
lg_model.fit(X_train, y_train)
dt_model = tree.DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

y_log_pred = lg_model.predict(X_test)
y_tree_pred = dt_model.predict(X_test)

print('F1-score для логистической регресии: ', round(metrics.f1_score(y_test, y_log_pred),3))
print('F1-score для дерева решений: ', round(metrics.f1_score(y_test, y_tree_pred),3))

F1-score для логистической регресии:  0.739
F1-score для дерева решений:  0.76


c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
# Задание 2.8
bg_model = BaggingClassifier(
    estimator=dt_model,
    n_estimators=1500,
    random_state=42
)
bg_model.fit(X_train, y_train)

y_bg_pred = bg_model.predict(X_test)
print('F1-score для бэггинга: ', round(metrics.f1_score(y_test, y_bg_pred),3))

F1-score для бэггинга:  0.824


### <center>3.Случайный лес<center>

In [26]:
boston_data = pd.read_csv('C:\\IDE\\data\\Block_6\\boston (1).csv')
boston_data.head()

,crim_rate,zn,business,river,nit_oxiden,rooms,age,dist,highways_index,tax,pup_per_teaс,lower,target
0,"0,00632",18,"2,31",0,"0,538","6,575","65,2","4,09",1,296,"15,3","4,98",24
1,"0,02731",0,"7,07",0,"0,469","6,421","78,9","4,9671",2,242,"17,8","9,14","21,6"
2,"0,02729",0,"7,07",0,"0,469","7,185","61,1","4,9671",2,242,"17,8","4,03","34,7"
3,"0,03237",0,"2,18",0,"0,458","6,998","45,8","6,0622",3,222,"18,7","2,94","33,4"
4,"0,06905",0,"2,18",0,"0,458","7,147","54,2","6,0622",3,222,"18,7","5,33","36,2"


In [30]:
boston_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crim_rate       506 non-null    object 
 1   zn              506 non-null    object 
 2   business        506 non-null    object 
 3   river           506 non-null    int64  
 4   nit_oxiden      506 non-null    object 
 5   rooms           506 non-null    object 
 6   age             506 non-null    object 
 7   dist            506 non-null    object 
 8   highways_index  506 non-null    int64  
 9   tax             506 non-null    int64  
 10  pup_per_teaс    506 non-null    object 
 11  lower           506 non-null    object 
 12  target          506 non-null    float64
dtypes: float64(1), int64(3), object(9)
memory usage: 51.5+ KB


In [31]:
def convert_commas_to_dots(df, columns):
    """
    Заменяет запятые на точки и удаляет пробелы в указанных столбцах,
    после чего преобразует их в числовой тип (float).
    """
    
    for col in columns:
        if col in df.columns:
            # Удаляем пробелы (если есть) и меняем запятую на точку
            df[col] = (df[col]
                            .astype(str) # Принудительно приводим к строке на случай смешанных типов
                            .str.replace(',', '.', regex=False))
            
            # Преобразуем в числовой тип, превращая ошибки в NaN (например, если встретится текст)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    return df

boston_data = convert_commas_to_dots(boston_data, boston_data.columns)

In [32]:
X = boston_data.drop('target', axis=1)
y = boston_data[['target']]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=13)

In [34]:
# Задание 3.4

print(y_train['target'].median())

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
print('MAE на линейной регрессии: ', round(metrics.mean_absolute_error(y_test, lr_pred),2))

tree_model = tree.DecisionTreeRegressor(random_state=13)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
print('MAE на дереве решений: ', round(metrics.mean_absolute_error(y_test, tree_pred),2))

21.55
MAE на линейной регрессии:  3.72
MAE на дереве решений:  2.84


In [36]:
rf_model_3 = RandomForestRegressor(n_estimators=3, random_state=13)
rf_model_10 = RandomForestRegressor(n_estimators=10, random_state=13)
rf_model_100 = RandomForestRegressor(n_estimators=100, random_state=13)
rf_model_500 = RandomForestRegressor(n_estimators=500, random_state=13)

rf_model_3.fit(X_train, y_train)
rf_model_10.fit(X_train, y_train)
rf_model_100.fit(X_train, y_train)
rf_model_500.fit(X_train, y_train)

pred_3_model = rf_model_3.predict(X_test)
pred_10_model = rf_model_10.predict(X_test)
pred_100_model = rf_model_100.predict(X_test)
pred_500_model = rf_model_500.predict(X_test)

print('MAE на моделе с 3 деревьями: ', round(metrics.mean_absolute_error(y_test, pred_3_model),2))
print('MAE на моделе с 10 деревьями: ', round(metrics.mean_absolute_error(y_test, pred_10_model),2))
print('MAE на моделе с 100 деревьями: ', round(metrics.mean_absolute_error(y_test, pred_100_model),2))
print('MAE на моделе с 500 деревьями: ', round(metrics.mean_absolute_error(y_test, pred_500_model),2))

c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\b

MAE на моделе с 3 деревьями:  2.93
MAE на моделе с 10 деревьями:  2.47
MAE на моделе с 100 деревьями:  2.26
MAE на моделе с 500 деревьями:  2.24


### <center>4.Случайный лес. Практика<center>

In [115]:
rain_data = pd.read_csv('C:\\IDE\\data\\Block_6\\weatherAUS.csv')
rain_data.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [116]:
# Задание 4.1

rain_data.isnull().sum().sum()

343248

In [117]:
# Задание 4.2

missing_percentage = rain_data.isnull().mean() * 100
print(missing_percentage)

rain_data = rain_data.drop(['Evaporation', 'Sunshine', 'Cloud3pm'], axis=1)

Date              0.000000
Location          0.000000
MinTemp           1.020899
MaxTemp           0.866905
Rainfall          2.241853
Evaporation      43.166506
Sunshine         48.009762
WindGustDir       7.098859
WindGustSpeed     7.055548
WindDir9am        7.263853
WindDir3pm        2.906641
WindSpeed9am      1.214767
WindSpeed3pm      2.105046
Humidity9am       1.824557
Humidity3pm       3.098446
Pressure9am      10.356799
Pressure3pm      10.331363
Cloud9am         38.421559
Cloud3pm         40.807095
Temp9am           1.214767
Temp3pm           2.481094
RainToday         2.241853
RainTomorrow      2.245978
dtype: float64


In [118]:
# Задание 4.3

rain_data['RainToday'] = rain_data['RainToday'].apply(lambda x: 1 if x == 'Yes' else 0)
rain_data['RainTomorrow'] = rain_data['RainTomorrow'].apply(lambda x: 1 if x == 'Yes' else 0)

print(round(rain_data['RainToday'].mean(),2))

0.22


In [119]:
# Задание 4.4

rain_data['Month'] = pd.to_datetime(rain_data['Date']).dt.month
rain_data = rain_data.drop('Date', axis=1)

rain_month = rain_data.groupby('Month')['RainToday'].sum()
print(rain_month)

Month
1     2447
2     2176
3     2831
4     2451
5     2901
6     3267
7     3189
8     2978
9     2600
10    2321
11    2415
12    2304
Name: RainToday, dtype: int64


In [120]:
# Задание 4.5

categoricals = ['Month', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']

rain_data = pd.get_dummies(rain_data, columns=categoricals)

print(rain_data.columns)

Index(['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am',
       'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am',
       'Pressure3pm',
       ...
       'WindDir3pm_NNW', 'WindDir3pm_NW', 'WindDir3pm_S', 'WindDir3pm_SE',
       'WindDir3pm_SSE', 'WindDir3pm_SSW', 'WindDir3pm_SW', 'WindDir3pm_W',
       'WindDir3pm_WNW', 'WindDir3pm_WSW'],
      dtype='object', length=124)


In [122]:
# Задание 4.6
rain_data_clean = rain_data.dropna()

X = rain_data_clean.drop('RainTomorrow', axis=1)
y = rain_data_clean['RainTomorrow']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=31)

print(round(y_test.mean(),2))

0.23


In [123]:
# Задание 4.7

# Фиксируем случайность
np.random.seed(31)

# Выделяем признак MinTemp, принудительно переводя в массив numpy
min_temp_values = rain_data['MinTemp'].to_numpy(dtype=float)

# ИСКЛЮЧАЕМ ПРОПУСКИ: оставляем только реальные числа
min_temp_values = min_temp_values[~np.isnan(min_temp_values)]


# Длина выборки
n = len(min_temp_values)

# Список для хранения средних значений каждой из 1000 выборок
bootstrap_means = []

# Генерируем 1000 бутстреп-выборок
for _ in range(1000):
    # Генерируем случайные индексы такого же объема с помощью np.random.randint
    indices = np.random.randint(0, n, size=n)
    
    # Извлекаем элементы по сгенерированным индексам
    sample = min_temp_values[indices]
    
    # Вычисляем среднее значение для текущей выборки и сохраняем
    bootstrap_means.append(sample.mean())

# Вычисляем стандартное отклонение полученных средних значений
std_of_means = np.std(bootstrap_means)

# Выводим ответ, округленный до двух знаков после запятой
print(f"Оценка стандартного отклонения для среднего значения: {std_of_means:.2f}")


Оценка стандартного отклонения для среднего значения: 0.02


In [126]:
# Задание 4.8

lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
print('Roc-auc score для линейной регрессии: ', round(metrics.roc_auc_score(y_test, y_pred),2))

Roc-auc score для линейной регрессии:  0.73


c:\Users\Smoking Shop\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Задание 4.9

params = {'max_leaf_nodes': list(range(2, 10)), 'min_samples_split': [2, 3, 4], 'max_depth': [5,7,9,11]}

grid_serch = GridSearchCV(tree.DecisionTreeClassifier(random_state=42), param_grid=params, cv=3)
grid_serch.fit(X_train, y_train)
grid_serch.predict(X_test)
print(grid_serch.best_params_)

dt_model = tree.DecisionTreeClassifier(max_depth=5, max_leaf_nodes=8, min_samples_split=2, random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)
print('Roc-auc score для дерева решений: ', round(metrics.roc_auc_score(y_test, y_pred),2))

{'max_depth': 5, 'max_leaf_nodes': 8, 'min_samples_split': 2}
Roc-auc score для дерева решений:  0.71


In [133]:
# Задание 4.10

rf_model = RandomForestClassifier(n_estimators=100, random_state=31)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('Roc-auc score для рандомного леса: ', round(metrics.roc_auc_score(y_test, y_pred),2))

Roc-auc score для рандомного леса:  0.74


In [136]:
# Задание 4.11

params = {'max_features': [ 4, 5, 6, 7], 'min_samples_leaf': [3, 5, 7, 9, 11], 'max_depth': [5, 10, 15]}

grid_serch = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=31), param_grid=params, cv=3)
grid_serch.fit(X_train, y_train)
grid_serch.predict(X_test)
print(grid_serch.best_params_)

rf_model = RandomForestClassifier(n_estimators=100,max_depth=15, max_features=7, min_samples_leaf=3,random_state=31)
rf_model.fit(X_train,y_train)
y_pred = rf_model.predict(X_test)
print('Roc-auc score для рандомного леса: ', round(metrics.roc_auc_score(y_test, y_pred),2))

{'max_depth': 15, 'max_features': 7, 'min_samples_leaf': 3}
Roc-auc score для рандомного леса:  0.7


In [137]:
# Задание 4.12

# Получаем коэффициенты важности
importances = rf_model.feature_importances_

# Создаем DataFrame для наглядности
feature_importance_df = pd.DataFrame({
    'Признак': X_train.columns,
    'Важность': importances
})

# Сортируем по убыванию и берем топ-3
top_3_features = feature_importance_df.sort_values(by='Важность', ascending=False).head(3)

print("Топ-3 самых важных признака:")
print(top_3_features.to_string(index=False))

Топ-3 самых важных признака:
    Признак  Важность
Humidity3pm  0.249488
   Rainfall  0.081319
   Cloud9am  0.065026


### <center>5.Бустинг<center>
#### <center>Adaboost<center>
#### <center>Градиентный бустинг<center>
#### XGBOOST и CATBOOST
### <center>6.Градиентный бустинг. Практика<center>

In [138]:
air_data = pd.read_csv('C:\\IDE\\data\\Block_6\\AirPass.csv')
air_data.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [139]:
air_data = air_data.drop('Unnamed: 0', axis=1)
air_data.head()

,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [140]:
# Задание 6.1

air_data.isnull().sum()

id                                     0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             310
satisfaction                           0
dtype: int64

In [142]:
# Задание 6.2
median_value = air_data['Arrival Delay in Minutes'].median()
air_data['Arrival Delay in Minutes'] = air_data['Arrival Delay in Minutes'].fillna(median_value)
print(round(air_data['Arrival Delay in Minutes'].mean(),2))

15.13
